### RandomForestClassifier로 다음 LOT의 구역별 후보 비교

- `1`은 아래 통계 규칙과 공통 움직임 검토를 반영한 이상 후보임.
  - 이 노트북은 그 라벨을 얼마나 잘 재현하는지 확인함.
  - 실제 고장 정답이나 고장 확률로 해석하지 않음.
- 같은 구역의 온도 `TC`와 출력 `OP`를 한 모델에 넣음.
  - `Z4`에는 `CP-Z4`도 함께 넣음.
  - `CLN`과 `Z9`는 각각 한 센서, `COOLING`은 `TC-OUT1`과 `TC-OUT2`를 사용함.
  - 다른 구역의 결과를 합쳐 행 전체에 `1`을 붙이지 않음.
- `1 파일` → `2 기록` → `3 입력` → `4 분할` → `5 학습` → `6 비교` → `7 그래프` → `8 설정 저장` 순서임.
- 기존 구역 라벨은 같은 LOT의 평균에서 3표준편차를 넘긴 후보를 바탕으로 함.
  - 검토한 공통 움직임만 후보에서 뺐고, 모든 1을 정비 기록으로 확인한 것은 아님.
  - 1분 안에 남은 후보 원기록이 하나라도 있으면 그 분의 구역 라벨은 1임.
  - 분 전체가 계속 이상이었다는 뜻이 아니며, 고장 정답으로 사용하지 않음.


### 1. 파일과 실행 옵션

- `data/processed/`에 `cal01_base_v5.csv` (1분 집계)를 넣음.
  - 아래 `DATA_FILE`의 파일명만 바꾸고 전체 실행하면 됨.
  - 노트북의 위치는 옮기지 않음.
- 아래 원본 3개도 같은 폴더에 준비함.
  - `G-02_조업이벤트.csv`
  - `G-02_정비이력.csv`
  - `G-01_정기수리캘린더.csv`
  - `data/raw/T-CR1-CAL01_온도.csv`가 있으면 `data/raw/`에서 읽고, 없으면 `data/`에서 읽음.
- `CLASS_WEIGHT=None`은 각 학습 행에 같은 무게를 줌.
  - `"balanced"`는 적은 라벨에 더 큰 무게를 줌.
  - 행을 복제하거나 삭제하지 않으며 Dummy는 원래 Train 라벨 비율을 사용함.
- `PROBABILITY_THRESHOLD=0.5`는 예측값을 후보 `1`로 정하는 기준임.
  - 실행 전에 정하고 Test 결과를 보고 바꾸지 않음.
- `RUN_FIRST_LOT=True`면 001 내부의 시간순 비교도 추가함.
  - `True`와 `False`는 따옴표 없이 적음.
- `DUMMY_REPEATS=100`은 무작위 Dummy를 100번 비교한다는 뜻임.
  - 1 이상의 정수이며 1번만 돌리면 표준편차와 분산은 비워 둠.
- 파일이나 옵션을 바꾸면 첫 셀부터 전체 실행하고 마지막 저장 셀까지 마침.

- 결과표는 모든 구역을 보여줌.
  - 그림도 모든 LOT와 구역을 Figure 하나에 모음.

In [ ]:
DATA_FILE = "cal01_base_v5.csv"  # 1분 집계 v5. 다른 파일도 이름만 바꿔 실행함.
RUN_FIRST_LOT = False  # True면 001 안에서 시간 순서 60/20/20도 실행함
RUN_NO_HISTORY_LOT = True  # 이전 동일 강종이 없으면 해당 LOT 내부 시간 분할을 별도 실행함.

import os
import json
import hashlib
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import sklearn
from matplotlib import font_manager, rcParams
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib.patches import Patch
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from IPython.display import Markdown

ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)

ENC = "utf-8-sig"
PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
BASE_CSV = os.path.join(PROCESSED_DIR, DATA_FILE)
files = {
    "base": BASE_CSV,
    "event": "G-02_조업이벤트.csv",
    "maintenance": "G-02_정비이력.csv",
    "repair_plan": "G-01_정기수리캘린더.csv",
}
# 노션 공통 코드와 같은 폴더 선택임
DATA_DIR = os.path.join(ROOT, "data")
if os.path.isfile(os.path.join(DATA_DIR, "raw", "T-CR1-CAL01_온도.csv")):
    DATA_DIR = os.path.join(DATA_DIR, "raw")

for key in ["event", "maintenance", "repair_plan"]:
    files[key] = os.path.join(DATA_DIR, files[key])

input_hashes = {}
for key, path in files.items():
    with open(path, "rb") as source_file:
        input_hashes[key] = hashlib.sha256(source_file.read()).hexdigest()

RUN_ID = os.path.splitext(DATA_FILE)[0] + "__" + input_hashes["base"][:8] + "__" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUT_DIR = os.path.join(PROCESSED_DIR, "sayletskin", "random_forest_classifier", RUN_ID)
FIG_DIR = os.path.join(ROOT, "figures", "sayletskin", "random_forest_classifier", RUN_ID)
os.makedirs(OUT_DIR)
os.makedirs(FIG_DIR)

font_names = []
for installed_font in font_manager.fontManager.ttflist:
    font_names.append(installed_font.name)
for font_name in ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]:
    if font_name in font_names:
        rcParams["font.family"] = font_name
        break
rcParams["axes.unicode_minus"] = False
rcParams["figure.dpi"] = 110

# 난수 번호를 고정해 같은 입력에서 같은 결과를 얻음
SEED = 42
N_ESTIMATORS = 100  # 나무별 라벨 1 예측값을 평균냄
MAX_DEPTH = 12  # 나무가 조건을 나누는 깊이를 제한함
MIN_SAMPLES_LEAF = 5  # 나무의 마지막 판단 칸에 필요한 최소 학습 기록 수
CLASS_WEIGHT = None  # None은 같은 무게. "balanced"는 적은 라벨에 더 큰 무게를 줌
PROBABILITY_THRESHOLD = 0.5  # 라벨 1의 예측값이 이 값 이상이면 구역 후보 1
DUMMY_REPEATS = 100  # 1 이상의 정수. 같은 평가 행에서 난수 시드만 바꿈
MAX_GAP_SECONDS = 15  # 기록이 이 초보다 벌어지면 과거 계산을 다시 시작함
MIN_PAST_ROWS = 10  # 직전 60초 안에서 필요한 최소 관측 수
MIN_REFERENCE_ROWS = 256  # v3에서 강종과 작업 조건별 Train 기준에 필요한 최소 행 수
MIN_REFERENCE_MINUTES = 5  # Train 기준에 필요한 서로 다른 관측 분의 수
MIN_MINUTE_REFERENCE_ROWS = 5  # v4 Train 기준에 필요한 유효한 1분 행 수

TRAIN_LOTS = ["240001", "240024", "240037", "240038", "240044", "240058", "240061", "240069", "240091"]
VALID_LOTS = ["240108", "240114"]
TEST_LOTS = ["240117", "240121", "240133"]
LOTS = TRAIN_LOTS + VALID_LOTS + TEST_LOTS
groups = {
    "CLN": ["CLN-IN"],
    "Z1": ["TC-Z1", "OP-Z1"],
    "Z2": ["TC-Z2", "OP-Z2"],
    "Z3": ["TC-Z3", "OP-Z3"],
    "Z4": ["TC-Z4", "OP-Z4", "CP-Z4"],
    "Z5": ["TC-Z5", "OP-Z5"],
    "Z6": ["TC-Z6", "OP-Z6"],
    "Z7": ["TC-Z7", "OP-Z7"],
    "Z8": ["TC-Z8", "OP-Z8"],
    "Z9": ["TC-Z9"],
    "COOLING": ["TC-OUT1", "TC-OUT2"],
}

if type(RUN_FIRST_LOT) is not bool:
    raise ValueError("RUN_FIRST_LOT은 True 또는 False를 따옴표 없이 입력함.")
if type(DUMMY_REPEATS) is not int or DUMMY_REPEATS < 1:
    raise ValueError("DUMMY_REPEATS는 1 이상의 정수를 입력함.")

group_titles = {
    "CLN": "세정 CLN-IN",
    "Z1": "건조 Z1",
    "Z2": "건조 Z2",
    "Z3": "가열 Z3",
    "Z4": "가열 Z4와 CP",
    "Z5": "가열 Z5",
    "Z6": "가열 Z6",
    "Z7": "과시효 Z7",
    "Z8": "과시효 Z8",
    "Z9": "과시효 Z9",
    "COOLING": "냉각 OUT1과 OUT2",
}
GRADE_GROUPS = ["Z3", "Z4", "Z5", "Z6", "Z7", "Z8", "Z9", "COOLING"]
SENSORS = []
for group_sensors in groups.values():
    SENSORS.extend(group_sensors)
META = ["ROW_ID", "LOT_NO", "MEAS_DT", "GRADE", "EXCLUDED", "EXCLUSION_REASON"]
LABEL_COLUMNS = []
for group in groups:
    LABEL_COLUMNS.append(group + "__LABEL")
META = META + LABEL_COLUMNS
method_names = {
    "fixed_9_2_3": "고정 9/2/3",
    "same_grade_history": "이전 동일 강종",
    "all_history": "이전 모든 강종",
    "first_lot_60_20_20": "001 시간 60/20/20",
    "within_lot_60_20_20": "같은 강종 없음: LOT 내부 60/20/20",
}
print("입력:", BASE_CSV)
print("결과:", OUT_DIR)
print("그림:", FIG_DIR)


### 2. CSV와 조업 기록 읽기

- `LOT_NO`는 관측 LOT, `MEAS_DT`는 표시 시각, `GRADE`는 강종임.
  - 원본 기록의 `MEAS_DT`는 실제 기록 시각임.
  - 1분 입력의 `MEAS_DT`는 분 시작 시각임.
  - 실제 첫 기록과 마지막 기록은 `OBS_START`와 `OBS_END`에 남음.
- 구역마다 `Z3__LABEL`처럼 별도의 라벨을 읽음.
  - v1과 v2의 행 전체 라벨을 구역 라벨로 복사하지 않음.
- 원본 기록에는 그 시각보다 앞서거나 같은 시각의 가장 최근 강종변경을 연결함.
  - `direction="backward"`가 이 방향을 정하는 옵션임.
- `CONTEXT`는 정비 작업이나 정기수리 계획과 시간이 겹치는지 나타냄.
  - 정비 종료 기록까지 사용하므로 정비 연결은 사후 확인임.
  - 정비와 겹친다는 이유만으로 고장이나 정지라고 판단하지 않음.
- 1분 CSV는 집계 전에 연결한 기록을 그대로 읽음.
  - `AVAILABLE_AT`은 해당 분의 값을 사용할 수 있는 시각임.
- `EXCLUDED=True`인 행도 결과 표에는 남김.
  - 학습과 모델 판정은 보류하고 결과를 `0`으로 채우지 않음.
  - 058의 정기수리 전 정지와 061의 정기수리 후 재가동은 팀 판단을 유지함.
- CSV 안의 `DATA_KIND`는 원본 기록인지 1분 집계인지 나타냄.
  - 1분 집계의 `FEATURE_PROFILE`은 과거 60초 특징을 먼저 만들고 분 안에서 평균낸 방식이어야 함.
  - 필수 컬럼이 없거나 다른 계산 방식이면 실행을 멈추고 이유를 표시함.
- `Z3__LABEL`은 기존 구역 라벨이고 `Z3__FLAG`는 모델이 계산한 후보임.
  - `1`은 후보, `0`은 해당 기준에서 후보가 아니라는 뜻임.
  - 고장이나 정상을 확정한 값은 아님.
  - 라벨이 빈 평가 행도 입력과 학습 조건이 맞으면 모델 후보를 계산함.
  - 라벨이 없는 평가 행은 지표 계산에서 빼며, 모델 판정 보류와 구분함.
  - 구역 라벨에 `0`, `1`, 빈값 외의 값이 있으면 실행을 멈춤.

- v5의 `__STAT_LABEL`은 공통 움직임 해제 전의 통계 후보임.
  - `__DOMAIN_LABEL`은 별도로 남긴 검토 결과이며 미검토는 빈칸임.
  - 이번 비교 대상은 기존처럼 `__LABEL`, 즉 검토를 반영한 통계 후보임.
  - 새 라벨과 판단 이유는 모델 입력에 넣지 않음.
  - 도메인 검토 라벨만으로 학습한 실험이라고 해석하지 않음.


In [ ]:
# 결과를 다시 쓰기 전에 이전 실행의 완료 표시를 지움.
completion_file = os.path.join(OUT_DIR, "run_config.json")
if os.path.isfile(completion_file):
    os.remove(completion_file)

# 컬럼 이름을 먼저 읽어 원본 기록인지 1분 집계인지 구분함
header = pd.read_csv(BASE_CSV, encoding=ENC, nrows=0).columns.tolist()
# 1분 집계 열이 있는데 시각 열이 빠지면 원본 기록으로 잘못 읽지 않음.
minute_suffixes = ("__VALUE", "__DIFF_PER_SEC", "__DEV_60S", "__STD_60S",
                   "__MODEL_ELIGIBLE", "__FEATURE_COUNT", "__GRADE_CHANGE_LOG_MIN")
has_minute_fields = "BIN_END" in header or "FEATURE_PROFILE" in header
has_minute_fields = has_minute_fields or any(column.endswith(minute_suffixes) for column in header)
if "BIN_START" not in header and has_minute_fields:
    raise ValueError("1분 집계 열이 있는데 BIN_START가 없습니다. CSV 생성 코드를 확인해주세요.")
DATA_KIND = "minute" if "BIN_START" in header else "raw"
CONTEXT_META = ["EVT_ID", "EVT_DT", "BEF_VAL", "AFT_VAL", "MINUTES_SINCE_GRADE_CHANGE",
                "EVENT_MATCH", "MAINT_OVERLAP", "MAINT_WORK_IDS", "REPAIR_PLAN_IDS",
                "CONTEXT", "OPERATING_NOTE"]
TIME_META = ["OBS_START", "OBS_END", "AVAILABLE_AT"]
read_columns = META + SENSORS
for column in ["DATA_KIND", "FEATURE_PROFILE"]:
    if column in header:
        read_columns.append(column)
minute_feature_columns = []

if DATA_KIND == "minute":
    read_columns = read_columns + CONTEXT_META + TIME_META + ["BIN_START", "BIN_END", "SEGMENT", "ROW_COUNT"]
    for sensor in SENSORS:
        for feature in ["VALUE", "DIFF_PER_SEC", "DEV_60S", "STD_60S"]:
            minute_feature_columns.append(sensor + "__" + feature)
    for group in groups:
        minute_feature_columns.append(group + "__MODEL_ELIGIBLE")
        minute_feature_columns.append(group + "__FEATURE_COUNT")
        if group in GRADE_GROUPS:
            minute_feature_columns.append(group + "__GRADE_CHANGE_LOG_MIN")
    missing_features = [column for column in minute_feature_columns if column not in header]
    if missing_features:
        raise ValueError("1분 입력에 필요한 파생 열이 없습니다: " + ", ".join(missing_features))
    read_columns = read_columns + minute_feature_columns

# 예전 v1, v2에는 구역 라벨이 없음. 행 전체 라벨을 구역 정답으로 복사하지 않음
available_columns = []
for column in read_columns:
    if column in header:
        available_columns.append(column)

df = pd.read_csv(
    BASE_CSV, encoding=ENC, usecols=available_columns,
    dtype={"ROW_ID": "string", "LOT_NO": "string", "GRADE": "string"}, low_memory=False,
)
# 파일명은 자유롭게 바꿔도 되며, 계산 방식은 파일 안의 표시로 확인함.
if "DATA_KIND" in df:
    declared_kind = df["DATA_KIND"].astype("string").str.strip()
    if not declared_kind.eq(DATA_KIND).fillna(False).all():
        raise ValueError("DATA_KIND와 시간 열 구성이 다릅니다. 원본과 1분 집계를 섞지 마세요.")
elif DATA_KIND == "minute":
    raise ValueError("1분 CSV에는 DATA_KIND=minute이 필요합니다.")

if DATA_KIND == "minute":
    if "FEATURE_PROFILE" not in df:
        raise ValueError("1분 CSV에는 FEATURE_PROFILE이 필요합니다.")
    profile = df["FEATURE_PROFILE"].astype("string").str.strip()
    if not profile.eq("raw60s_groupmean_1min").fillna(False).all():
        raise ValueError("1분 CSV의 모든 FEATURE_PROFILE은 raw60s_groupmean_1min이어야 합니다.")

df["MEAS_DT"] = pd.to_datetime(df["MEAS_DT"])
df["EXCLUDED"] = ~df["EXCLUDED"].astype("string").str.strip().str.lower().isin(["false", "0", "0.0"])
df["EXCLUSION_REASON"] = df["EXCLUSION_REASON"].fillna("")
df["GRADE"] = df["GRADE"].fillna("미기록")

for column in LABEL_COLUMNS:
    if column not in df:
        df[column] = np.nan
    label_values = pd.to_numeric(df[column], errors="coerce")
    has_label = df[column].astype("string").str.strip().fillna("").ne("")
    if (has_label & ~label_values.isin([0, 1])).any():
        raise ValueError(column + "에는 0, 1, 빈값만 넣어주세요.")
    df[column] = label_values.where(label_values.isin([0, 1]))

for sensor in SENSORS:
    df[sensor] = pd.to_numeric(df[sensor], errors="coerce").replace([np.inf, -np.inf], np.nan)

if DATA_KIND == "raw":
    events = pd.read_csv(files["event"], encoding=ENC)
    events = events.loc[events["PLANT_CD"].eq("CR1") & events["EVT_TYPE"].eq("강종변경")].copy()
    events["EVT_DT"] = pd.to_datetime(events["EVT_DT"])
    events = events.sort_values("EVT_DT")
    # backward는 관측 시각보다 앞서거나 같은 시각의 가장 최근 강종변경을 연결함
    # 미래 조업이벤트를 끌어오지 않음
    df = pd.merge_asof(
        df.sort_values("MEAS_DT"),
        events[["EVT_ID", "EVT_DT", "BEF_VAL", "AFT_VAL"]],
        left_on="MEAS_DT", right_on="EVT_DT", direction="backward",
    )
    df = df.sort_values(["LOT_NO", "MEAS_DT"], kind="stable").reset_index(drop=True)
    df["MINUTES_SINCE_GRADE_CHANGE"] = (df["MEAS_DT"] - df["EVT_DT"]).dt.total_seconds() / 60
    df["EVENT_MATCH"] = df["GRADE"].eq(df["AFT_VAL"]).fillna(False) & df["EVT_DT"].notna()

    maintenance = pd.read_csv(files["maintenance"], encoding=ENC)
    maintenance = maintenance.loc[maintenance["EQP_TAG"].eq("P1-CR1-CAL01")].copy()
    for column in ["STRT_DT", "END_DT"]:
        maintenance[column] = pd.to_datetime(maintenance[column])
    df["MAINT_WORK_IDS"] = ""
    for work in maintenance.itertuples(index=False):
        overlap = df["MEAS_DT"].ge(work.STRT_DT) & df["MEAS_DT"].lt(work.END_DT)
        df.loc[overlap, "MAINT_WORK_IDS"] += work.WO_NO + "|"
    df["MAINT_WORK_IDS"] = df["MAINT_WORK_IDS"].str.rstrip("|")
    df["MAINT_OVERLAP"] = df["MAINT_WORK_IDS"].ne("")

    repair = pd.read_csv(files["repair_plan"], encoding=ENC)
    repair = repair.loc[repair["PLANT_CD"].eq("CR1")].copy()
    repair["STRT_DT"] = pd.to_datetime(repair["STRT_DT"])
    repair["END_EXCLUSIVE"] = pd.to_datetime(repair["END_DT"]).dt.normalize() + pd.Timedelta(days=1)
    df["REPAIR_PLAN_IDS"] = ""
    for plan in repair.itertuples(index=False):
        overlap = df["MEAS_DT"].ge(plan.STRT_DT) & df["MEAS_DT"].lt(plan.END_EXCLUSIVE)
        df.loc[overlap, "REPAIR_PLAN_IDS"] += plan.PLAN_ID + "|"
    df["REPAIR_PLAN_IDS"] = df["REPAIR_PLAN_IDS"].str.rstrip("|")

    df["CONTEXT"] = "작업 기록 없음"
    df.loc[df["MAINT_OVERLAP"], "CONTEXT"] = "정비 작업 기록과 겹침"
    df.loc[df["REPAIR_PLAN_IDS"].ne(""), "CONTEXT"] = "정기수리 계획 기간"
    df["OPERATING_NOTE"] = "별도 운전 상태 기록 없음"
    df.loc[df["EXCLUDED"], "OPERATING_NOTE"] = df.loc[df["EXCLUDED"], "EXCLUSION_REASON"]
    df.loc[df["EXCLUDED"] & df["LOT_NO"].eq("240058"), "OPERATING_NOTE"] = "정기수리를 위해 온도를 내림, 팀 확정"
    df.loc[df["EXCLUDED"] & df["LOT_NO"].eq("240061"), "OPERATING_NOTE"] = "정기수리 후 재가동하며 온도를 올림, 팀 확정"


    # 원본 기록 한 행은 해당 시각에 사용할 수 있음
    df["OBS_START"] = df["MEAS_DT"]
    df["OBS_END"] = df["MEAS_DT"]
    df["AVAILABLE_AT"] = df["MEAS_DT"]
else:
    # v4는 원본 기록 때 연결한 강종과 작업 기록을 유지함
    # 분 끝 시각으로 다시 연결하면 다음 강종이 붙을 수 있어 다시 연결하지 않음
    for column in TIME_META + ["BIN_START", "BIN_END", "EVT_DT"]:
        df[column] = pd.to_datetime(df[column])
    for column in ["EVENT_MATCH", "MAINT_OVERLAP"]:
        df[column] = df[column].astype("string").str.strip().str.lower().isin(["true", "1", "1.0"])
    for column in ["MAINT_WORK_IDS", "REPAIR_PLAN_IDS", "OPERATING_NOTE"]:
        df[column] = df[column].fillna("")
    for column in minute_feature_columns:
        if column.endswith("__MODEL_ELIGIBLE"):
            df[column] = df[column].astype("string").str.strip().str.lower().isin(["true", "1", "1.0"])
        else:
            df[column] = pd.to_numeric(df[column], errors="coerce").replace([np.inf, -np.inf], np.nan)
    df = df.sort_values(["LOT_NO", "MEAS_DT", "SEGMENT"], kind="stable").reset_index(drop=True)
    MIN_REFERENCE_ROWS = MIN_MINUTE_REFERENCE_ROWS

context_summary = df.groupby(["LOT_NO", "GRADE", "CONTEXT", "OPERATING_NOTE"], dropna=False).agg(
    ROWS=("ROW_ID", "size"), START=("OBS_START", "min"), END=("OBS_END", "max"),
).reset_index()
context_summary.to_csv(os.path.join(OUT_DIR, "operating_context.csv"), index=False, encoding=ENC)
display(context_summary.rename(columns={
    "LOT_NO": "LOT", "GRADE": "강종", "CONTEXT": "작업 기록과의 관계",
    "OPERATING_NOTE": "팀 판단 또는 상태 기록", "ROWS": "행수", "START": "첫 시각", "END": "마지막 시각",
}))


### 3. 같은 구역의 온도와 출력을 함께 입력

| 입력 | 의미 |
| --- | --- |
| `VALUE` | 현재 센서값 |
| `DIFF_PER_SEC` | 직전 관측과의 차이를 실제 지난 초로 나눈 값 |
| `DEV_60S` | 현재값에서 직전 60초 평균을 뺀 값 |
| `STD_60S` | 직전 60초 값이 흔들린 정도 |

- `closed="left"`는 현재값을 과거 60초 계산에서 뺌.
- `min_periods=10`은 직전 60초 안에 유효한 기록이 10개 이상이면 계산함.
  - 60초 전체가 관측됐다는 뜻은 아님.
- LOT, 기록 공백, 제외 여부, 강종변경, 작업 조건이 바뀌면 과거 계산을 다시 시작함.
- 1분 CSV는 위 특징을 원본 기록에서 계산한 뒤 1분씩 평균 낸 값을 읽음.
  - 1분 평균값에 초 단위 계산을 다시 적용하지 않음.
  - 짧게 관측되었거나 입력이 빠진 분은 판정을 보류함.
- 강종에 반응하는 구역에는 강종변경 후 지난 시간도 넣음.
  - `log1p`로 긴 시간을 줄여 표현함.
- 라벨과 라벨 근거, LOT 전체 점수, 중앙 60%로 만든 온도 특징은 입력에서 뺌.

In [ ]:
if DATA_KIND == "raw":
    seconds = df["MEAS_DT"].diff().dt.total_seconds()
    breaks = df["LOT_NO"].ne(df["LOT_NO"].shift()).fillna(True)
    breaks = breaks | seconds.gt(MAX_GAP_SECONDS) | seconds.le(0)
    breaks = breaks | df["EXCLUDED"].ne(df["EXCLUDED"].shift())
    breaks = breaks | df["GRADE"].ne(df["GRADE"].shift())
    breaks = breaks | df["EVT_ID"].ne(df["EVT_ID"].shift()).fillna(True)
    breaks = breaks | df["CONTEXT"].ne(df["CONTEXT"].shift())
    df["SEGMENT"] = breaks.cumsum()
    features = {}

    for sensor in SENSORS:
        feature = pd.DataFrame(np.nan, index=df.index, columns=["VALUE", "DIFF_PER_SEC", "DEV_60S", "STD_60S"])
        for segment, block in df.loc[~df["EXCLUDED"]].groupby("SEGMENT", sort=False):
            values = pd.Series(block[sensor].to_numpy(), index=pd.DatetimeIndex(block["MEAS_DT"]))
            elapsed = values.index.to_series().diff().dt.total_seconds()
            past = values.rolling("60s", closed="left", min_periods=MIN_PAST_ROWS)
            feature.loc[block.index, "VALUE"] = values.to_numpy()
            feature.loc[block.index, "DIFF_PER_SEC"] = values.diff().div(elapsed).to_numpy()
            feature.loc[block.index, "DEV_60S"] = values.sub(past.mean()).to_numpy()
            feature.loc[block.index, "STD_60S"] = past.std().to_numpy()
        features[sensor] = feature.replace([np.inf, -np.inf], np.nan).astype("float32")

    group_inputs = {}
    group_valid = {}
    for group, sensors in groups.items():
        parts = []
        for sensor in sensors:
            parts.append(features[sensor].add_prefix(sensor + "__"))
        group_input = pd.concat(parts, axis=1)
        if group in GRADE_GROUPS:
            group_input["GRADE_CHANGE_LOG_MIN"] = np.log1p(df["MINUTES_SINCE_GRADE_CHANGE"])
        group_inputs[group] = group_input.astype("float32")
        valid = np.isfinite(group_input.to_numpy()).all(axis=1) & ~df["EXCLUDED"].to_numpy()
        if group in GRADE_GROUPS:
            valid = valid & df["EVENT_MATCH"].to_numpy(dtype=bool)
        group_valid[group] = valid


else:
    # v4의 특징은 원본 기록에서 60초 계산을 마친 뒤 1분씩 평균 낸 값임
    # 1분 간격 자료에 60초 rolling을 다시 적용하지 않음
    group_inputs = {}
    group_valid = {}
    for group, sensors in groups.items():
        columns = []
        for sensor in sensors:
            for feature in ["VALUE", "DIFF_PER_SEC", "DEV_60S", "STD_60S"]:
                columns.append(sensor + "__" + feature)
        group_input = df[columns].copy()
        if group in GRADE_GROUPS:
            group_input["GRADE_CHANGE_LOG_MIN"] = df[group + "__GRADE_CHANGE_LOG_MIN"]
        group_inputs[group] = group_input.astype("float32")
        valid = np.isfinite(group_input.to_numpy()).all(axis=1)
        valid = valid & df[group + "__MODEL_ELIGIBLE"].to_numpy() & ~df["EXCLUDED"].to_numpy()
        if group in GRADE_GROUPS:
            valid = valid & df["EVENT_MATCH"].to_numpy(dtype=bool)
        group_valid[group] = valid

# 1분 행은 관측 시작과 값을 사용할 수 있는 분 끝을 분리해서 분할함
lot_info = df.groupby("LOT_NO", sort=False).agg(
    GRADE=("GRADE", "first"), START=("OBS_START", "min"),
    END=("AVAILABLE_AT", "max"), LAST_OBS=("OBS_END", "max"), TOTAL_ROWS=("ROW_ID", "size"),
)
group_table = []
for group, sensors in groups.items():
    valid = group_valid[group]
    label = df[group + "__LABEL"]
    group_table.append({
        "구역": group_titles[group], "센서": ", ".join(sensors),
        "입력 계산 가능": int(valid.sum()),
        "라벨 0": int((valid & label.eq(0)).sum()),
        "라벨 1": int((valid & label.eq(1)).sum()),
        "라벨 보류": int((valid & label.isna()).sum()),
    })
display(pd.DataFrame(group_table))


### 4. 앞선 LOT로 학습하고 다음 LOT 확인

| 방법 | 학습 | 확인할 LOT |
| --- | --- | --- |
| 고정 `9/2/3` | 앞 9개 LOT | Valid `108, 114`, Test `117, 121, 133` |
| 이전 동일 강종 | 해당 LOT보다 먼저 끝난 같은 강종 | 해당 LOT |
| 이전 모든 강종 | 해당 LOT보다 먼저 끝난 모든 강종 | 해당 LOT |

- `RUN_FIRST_LOT=True`면 001을 시간순 `60/20/20`으로도 나눔.
  - 1분 CSV에서 분할 경계를 걸치는 분은 사용하지 않음.
- 같은 121을 보더라도 이전 모든 강종 방식은 117까지 학습에 들어감.
  - 고정 `9/2/3`과 학습 이력이 다르므로 같은 실험이라고 부르지 않음.
- 비율 조정용 09번과 10번 CSV는 읽지 않음.
  - 무작위 분할 결과와 LOT 시간순 분할을 섞지 않음.
- `RUN_NO_HISTORY_LOT=True`이면 앞선 동일 강종이 없는 LOT도 따로 실행함.
  - 해당 LOT의 실제 시간 앞 60%를 Train, 다음 20%를 Valid, 뒤 20%를 Test로 둠.
  - 이름은 `같은 강종 없음: LOT 내부 60/20/20`임.
  - 고정 9/2/3과 학습 자료가 다르므로 공통 Test 비교에 섞지 않음.
- 내부 분할에서도 학습하지 못한 작업 조건은 보류함.
  - 분할했다고 모든 구간을 반드시 예측할 수 있는 것은 아님.

In [ ]:
# 결과를 다시 쓰기 전에 이전 실행의 완료 표시를 지움.
completion_file = os.path.join(OUT_DIR, "run_config.json")
if os.path.isfile(completion_file):
    os.remove(completion_file)

jobs = [{"case": "fixed_9_2_3", "method": "fixed_9_2_3", "roles": {"train": TRAIN_LOTS, "valid": VALID_LOTS, "test": TEST_LOTS}}]

for method in ["same_grade_history", "all_history"]:
    for lot in LOTS:
        history = []
        if lot in lot_info.index:
            previous = lot_info.loc[lot_info["END"].lt(lot_info.loc[lot, "START"])].sort_values("START")
            if method == "same_grade_history":
                previous = previous.loc[previous["GRADE"].eq(lot_info.loc[lot, "GRADE"])]
            history = previous.index.tolist()
        jobs.append({"case": method + "_" + lot, "method": method, "roles": {"train": history, "test": [lot]}})

if RUN_FIRST_LOT and not RUN_NO_HISTORY_LOT and "240001" in lot_info.index:
    first_start = lot_info.loc["240001", "START"]
    first_span = lot_info.loc["240001", "LAST_OBS"] - first_start
    jobs.append({
        "case": "first_lot_60_20_20",
        "method": "first_lot_60_20_20",
        "roles": {"train": ["240001"], "valid": ["240001"], "test": ["240001"]},
        "cuts": [first_start + first_span * 0.6, first_start + first_span * 0.8],
    })


# 동일 강종 기록이 없는 LOT도 자기 LOT의 앞부분으로 학습하는 실험을 따로 남김.
# 고정 9/2/3 Test를 학습에 섞지 않고, 독립된 CASE로 실행함.
if RUN_NO_HISTORY_LOT:
    no_history_lots = []
    for job in jobs:
        if job["method"] == "same_grade_history" and not job["roles"]["train"]:
            no_history_lots.extend(job["roles"]["test"])
    for lot in no_history_lots:
        if lot not in lot_info.index:
            continue
        start = lot_info.loc[lot, "START"]
        span = lot_info.loc[lot, "LAST_OBS"] - start
        jobs.append({
            "case": "within_lot_60_20_20_" + lot,
            "method": "within_lot_60_20_20",
            "roles": {"train": [lot], "valid": [lot], "test": [lot]},
            "cuts": [start + span * 0.6, start + span * 0.8],
        })

plan_rows = []
for job in jobs:
    job["indices"] = {}
    for role, lots in job["roles"].items():
        selected = df["LOT_NO"].isin(lots)
        if "cuts" in job:
            if role == "train":
                selected = selected & df["AVAILABLE_AT"].lt(job["cuts"][0])
            elif role == "valid":
                selected = selected & df["OBS_START"].ge(job["cuts"][0]) & df["AVAILABLE_AT"].lt(job["cuts"][1])
            else:
                selected = selected & df["OBS_START"].ge(job["cuts"][1])
        job["indices"][role] = df.index[selected].to_numpy()

    train_index = job["indices"]["train"]
    job["train_index"] = train_index[~df.loc[train_index, "EXCLUDED"].to_numpy()]
    job["status"] = "실행" if len(job["train_index"]) else "생략"
    job["reason"] = ""
    if not job["roles"]["train"]:
        job["reason"] = "이전 동일 강종 기록 없음" if job["method"] == "same_grade_history" else "이전 LOT 기록 없음"
    elif not len(job["train_index"]):
        job["reason"] = "학습에 사용할 제외되지 않은 행 없음"
    if job["method"] == "fixed_9_2_3":
        missing_lots = []
        for required_lot in LOTS:
            if required_lot not in lot_info.index:
                missing_lots.append(required_lot)
        if missing_lots:
            job["train_index"] = np.array([], dtype=int)
            job["status"] = "생략"
            job["reason"] = "고정 9/2/3에 필요한 LOT 없음: " + ",".join(missing_lots)

    for role, lots in job["roles"].items():
        for lot in lots:
            index = job["indices"][role]
            index = index[df.loc[index, "LOT_NO"].eq(lot).to_numpy(dtype=bool)]
            n_common = int((~df.loc[index, "EXCLUDED"]).sum())
            status, reason = job["status"], job["reason"]
            if not len(index):
                status, reason = "생략", "CSV에 해당 LOT 또는 시간 구간 없음"
            elif not n_common:
                status, reason = "생략", "계산에 사용할 제외되지 않은 행 없음"
            plan_rows.append({
                "CASE": job["case"],
                "METHOD": job["method"],
                "ROLE": role,
                "LOT_NO": lot,
                "GRADE": lot_info.loc[lot, "GRADE"] if lot in lot_info.index else "",
                "START": df.loc[index, "OBS_START"].min(),
                "END": df.loc[index, "AVAILABLE_AT"].max(),
                "STATUS": status,
                "REASON": reason,
                "TOTAL_ROWS": len(index),
                "COMMON_ROWS": n_common,
                "TRAIN_LOTS": ",".join(job["roles"]["train"]),
            })

split_plan = pd.DataFrame(plan_rows)
split_plan.to_csv(os.path.join(OUT_DIR, "split_plan.csv"), index=False, encoding=ENC)
display(split_plan.loc[split_plan["ROLE"].ne("train"), ["METHOD", "ROLE", "LOT_NO", "TRAIN_LOTS", "COMMON_ROWS", "STATUS", "REASON"]].rename(columns={
    "METHOD": "방법",
    "ROLE": "역할",
    "LOT_NO": "LOT",
    "TRAIN_LOTS": "학습 LOT",
    "COMMON_ROWS": "기존 제외를 뺀 행",
    "STATUS": "상태",
    "REASON": "이유",
}))


### 5. Train만 사용해서 학습

- 구역 라벨을 아는 Train 행만 학습함.
  - 평가 라벨이 비어 있어도 입력이 있으면 예측은 남김.
  - 라벨이 없는 평가 행은 지표 계산에서만 뺌.
- 같은 구역의 센서를 한 모델에 넣고 작업 조건별로 따로 학습함.
  - 예를 들어 `TC-Z3`와 `OP-Z3`를 함께 넣음.
- 강종에 반응하는 구역은 Train의 강종별 기준으로 온도와 출력 수준을 맞춤.
  - 중앙값을 빼고 가운데 50%의 범위로 나눔.
  - 범위가 0이면 표준편차를 쓰며 그것도 0이면 1로 나눔.
- 조건별 기준 계산에는 최소 기록 수를 둠.
  - v3는 유효한 256행 이상이고 기록이 있는 분도 5개 이상이어야 함.
  - 1분 CSV는 유효한 기록이 있는 서로 다른 분이 5개 이상이어야 함.
- 학습에 라벨 `0` 또는 `1`이 없으면 분류 학습을 보류함.
  - 이전에 학습하지 않은 강종이나 작업 조건도 보류함.
- `predict_proba`는 우리가 붙인 후보 라벨 `1`에 대한 예측값임.
  - 실제 고장 가능성을 뜻하지 않음.
- [RandomForestClassifier 옵션 설명](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)

In [ ]:
# 결과를 다시 쓰기 전에 이전 실행의 완료 표시를 지움.
completion_file = os.path.join(OUT_DIR, "run_config.json")
if os.path.isfile(completion_file):
    os.remove(completion_file)

reference_rows = []
fit_rows_path = os.path.join(OUT_DIR, "fit_rows.csv.gz")
# 학습에 넣으려던 행과 실제 학습 여부를 구역마다 저장함. 전체 목록을 모으지 않음.
# mode="a"는 파일 끝에 추가하며, header=False는 컬럼 이름을 반복해서 쓰지 않음.
# 추가 기록은 utf-8로 써서 파일 시작 표시가 중간에 끼지 않게 함.
pd.DataFrame(columns=[
    "ROW_ID", "LOT_NO", "MEAS_DT", "AVAILABLE_AT", "GRADE", "CONTEXT",
    "CASE", "GROUP", "Y", "STATUS",
]).to_csv(fit_rows_path, index=False, encoding=ENC, compression="gzip")
fit_distribution_rows = []
importance_rows = []
prediction_parts = []

for job in jobs:
    evaluation_index = np.array([], dtype=int)
    role_values = []
    for role in ["valid", "test"]:
        if role in job["indices"]:
            index = job["indices"][role]
            evaluation_index = np.concatenate([evaluation_index, index])
            role_values.extend([role] * len(index))

    result = df.loc[evaluation_index, META + CONTEXT_META + TIME_META].copy()
    result["CASE"] = job["case"]
    result["METHOD"] = job["method"]
    result["ROLE"] = role_values
    train_index = job["train_index"]

    for group, group_input in group_inputs.items():
        label_col = group + "__LABEL"
        score_col = group + "__PROBA"
        flag_col = group + "__FLAG"
        status_col = group + "__STATUS"
        result[score_col] = np.nan
        result[flag_col] = np.nan
        result[status_col] = "같은 작업 조건의 학습 기록 없음"
        result.loc[~group_valid[group][evaluation_index], status_col] = "입력 부족"
        if group in GRADE_GROUPS:
            result.loc[~result["EVENT_MATCH"], status_col] = "강종 연결 불일치"
        result.loc[result["EXCLUDED"], status_col] = "기존 제외"

        if not len(train_index):
            usable = ~result["EXCLUDED"] & group_valid[group][evaluation_index]
            result.loc[usable, status_col] = job["reason"] or "이전 LOT 없음"
            continue

        # 라벨은 Train에서만 학습에 사용함
        # 평가 라벨이 비어 있어도 입력이 있으면 예측은 남기고, 지표 계산에서만 뺌
        label_known = df[label_col].isin([0, 1]).to_numpy()
        usable_train = train_index[group_valid[group][train_index] & label_known[train_index]]
        usable_eval = evaluation_index[group_valid[group][evaluation_index]]
        if not len(usable_train):
            result.loc[usable_eval, status_col] = "학습에 쓸 구역 라벨과 입력 없음"
            continue
        grade_key = df["GRADE"] if group in GRADE_GROUPS else pd.Series("강종 구분 없음", index=df.index)

        for context in df.loc[usable_train, "CONTEXT"].unique():
            context_train = usable_train[df.loc[usable_train, "CONTEXT"].eq(context).to_numpy()]
            context_eval = usable_eval[df.loc[usable_eval, "CONTEXT"].eq(context).to_numpy()]
            if not len(context_eval):
                continue

            result.loc[context_eval, status_col] = "같은 강종의 학습 기준 없음"
            normalized_train = pd.DataFrame(np.nan, index=context_train, columns=group_input.columns)
            normalized_eval = pd.DataFrame(np.nan, index=context_eval, columns=group_input.columns)

            for grade in grade_key.loc[context_train].unique():
                reference_index = context_train[grade_key.loc[context_train].eq(grade).to_numpy()]
                target_index = context_eval[grade_key.loc[context_eval].eq(grade).to_numpy()]
                minutes = df.loc[reference_index, "MEAS_DT"].dt.floor("min").nunique()
                enough = len(reference_index) >= MIN_REFERENCE_ROWS and minutes >= MIN_REFERENCE_MINUTES
                if not enough:
                    result.loc[target_index, status_col] = "기준 표본 부족"
                    continue

                # 강종마다 온도와 출력의 수준이 달라 Train에서만 기준을 구함
                values = group_input.loc[reference_index]
                center = values.median()
                iqr = values.quantile(0.75) - values.quantile(0.25)
                scale = iqr.where(iqr.gt(0), values.std(ddof=0))
                scale = scale.where(scale.gt(0), 1.0)
                normalized_train.loc[reference_index] = (values - center) / scale
                normalized_eval.loc[target_index] = (group_input.loc[target_index] - center) / scale
                for feature_name in group_input.columns:
                    reference_rows.append({
                        "CASE": job["case"], "GROUP": group, "CONTEXT": context, "GRADE_KEY": grade,
                        "TRAIN_ROWS": len(reference_index), "OBSERVED_MINUTES": minutes,
                        "FEATURE": feature_name, "CENTER": center[feature_name], "SCALE": scale[feature_name],
                    })

            normalized_train = normalized_train.dropna().astype("float32")
            normalized_eval = normalized_eval.dropna().astype("float32")
            fit_index = normalized_train.index
            target_index = normalized_eval.index
            if not len(fit_index):
                continue

            y_train = df.loc[fit_index, label_col].astype(int)
            train_0 = int(y_train.eq(0).sum())
            train_1 = int(y_train.eq(1).sum())
            fit_distribution_rows.append({
                "CASE": job["case"], "GROUP": group, "CONTEXT": context,
                "TRAIN_ROWS": len(fit_index), "TRAIN_0": train_0, "TRAIN_1": train_1, "SEED": SEED,
            })
            fit_record = df.loc[fit_index, ["ROW_ID", "LOT_NO", "MEAS_DT", "AVAILABLE_AT", "GRADE", "CONTEXT"]].copy()
            fit_record["CASE"] = job["case"]
            fit_record["GROUP"] = group
            fit_record["Y"] = y_train.to_numpy()

            # 한 종류의 라벨만 있으면 0과 1을 구분하는 모델을 학습할 수 없다고 표시함
            if train_0 == 0 or train_1 == 0:
                missing = "0" if train_0 == 0 else "1"
                result.loc[target_index, status_col] = "학습할 라벨 " + missing + " 없음"
                fit_record["STATUS"] = "한 라벨만 있어 학습 보류"
                fit_record.to_csv(
                    fit_rows_path, mode="a", header=False, index=False,
                    encoding="utf-8", compression="gzip",
                )
                continue

            # max_features="sqrt": 각 분기에서 입력 컬럼 일부를 골라 나무들이 다르게 학습하게 함
            # bootstrap=True: Train 안에서 복원 추출함. Valid와 Test를 추출하지 않음
            model = RandomForestClassifier(
                n_estimators=N_ESTIMATORS,
                max_depth=MAX_DEPTH,
                min_samples_leaf=MIN_SAMPLES_LEAF,
                max_features="sqrt",
                bootstrap=True,
                class_weight=CLASS_WEIGHT,
                random_state=SEED,
                n_jobs=2,  # 나무 학습과 예측을 동시에 두 작업으로 나눠 실행함
            )
            model.fit(normalized_train.to_numpy(), y_train.to_numpy())
            fit_record["STATUS"] = "학습 완료"
            fit_record.to_csv(
                fit_rows_path, mode="a", header=False, index=False,
                encoding="utf-8", compression="gzip",
            )

            for feature_name, importance in zip(group_input.columns, model.feature_importances_):
                importance_rows.append({
                    "CASE": job["case"], "GROUP": group, "CONTEXT": context,
                    "FEATURE": feature_name, "IMPORTANCE": float(importance),
                })

            if len(target_index):
                class_position = model.classes_.tolist().index(1)
                probability = model.predict_proba(normalized_eval.to_numpy())[:, class_position]
                result.loc[target_index, score_col] = probability
                result.loc[target_index, flag_col] = (probability >= PROBABILITY_THRESHOLD).astype("int8")
                result.loc[target_index, status_col] = "계산 완료"

    prediction_parts.append(result)
    print(job["case"], "| 평가 기록", len(result))

predictions = pd.concat(prediction_parts, ignore_index=True)
fit_distribution = pd.DataFrame(fit_distribution_rows, columns=[
    "CASE", "GROUP", "CONTEXT", "TRAIN_ROWS", "TRAIN_0", "TRAIN_1", "SEED",
])
# 학습할 라벨이 없어도 컬럼 이름이 있는 빈 표를 저장함
references = pd.DataFrame(reference_rows, columns=[
    "CASE", "GROUP", "CONTEXT", "GRADE_KEY", "TRAIN_ROWS", "OBSERVED_MINUTES",
    "FEATURE", "CENTER", "SCALE",
])
importance = pd.DataFrame(importance_rows, columns=[
    "CASE", "GROUP", "CONTEXT", "FEATURE", "IMPORTANCE",
])
predictions.to_csv(os.path.join(OUT_DIR, "predictions.csv"), index=False, encoding=ENC, float_format="%.17g")
fit_distribution.to_csv(os.path.join(OUT_DIR, "fit_distribution.csv"), index=False, encoding=ENC)
references.to_csv(os.path.join(OUT_DIR, "train_reference.csv"), index=False, encoding=ENC)
importance.to_csv(os.path.join(OUT_DIR, "feature_importance.csv"), index=False, encoding=ENC)


### 6. 같은 구역과 같은 행에서 비교

- `RandomForestClassifier`가 판정했고 구역 라벨이 있는 행에서 비교함.
  - 라벨을 모르거나 판정하지 못한 행은 `0`으로 채우지 않음.
  - 놓친 `1`과 비교에서 빠진 `1`은 따로 셈.
- `Dummy 0 고정`은 모두 `0`으로 예측함.
- `Dummy 무작위`는 해당 구역과 작업 조건의 Train 라벨 비율로 예측함.
  - `DUMMY_REPEATS=100`이면 시드 `0~99`를 사용함.
  - 평가 라벨의 비율은 쓰지 않음.
- 학습 방식끼리는 같은 구역에서 함께 판정한 Test 행으로 비교함.
  - 공통 행만 남겨 빠진 행 수도 기록함.
- 예측 `1`이 없으면 `Precision`, 실제 구역 `1`이 없으면 `Recall`을 비워 둠.
  - 비교 행이 있지만 실제 `1`과 예측 `1`이 모두 없으면 `F1=0`임.
  - 이는 `zero_division=0`과 같으며 비교 행 자체가 없으면 `F1`도 비워 둠.
- 무작위 결과의 평균과 퍼짐은 같은 자료에서 시드만 바꾼 차이임.
  - 미래 성능의 신뢰구간을 뜻하지 않음.

### 6-1. 세 모델이 함께 비교할 행 선택

In [ ]:
from sklearn.dummy import DummyClassifier

RANDOM_SEEDS = list(range(DUMMY_REPEATS))
TEST_LOTS = ["240117", "240121", "240133"]
method_names = {
    "fixed_9_2_3": "고정 9/2/3",
    "same_grade_history": "이전 동일 강종",
    "all_history": "이전 모든 강종",
    "first_lot_60_20_20": "001 시간 60/20/20",
    "within_lot_60_20_20": "같은 강종 없음: LOT 내부 60/20/20",
}

# 더미의 Train 분포도 연결되는 행으로 세 모델의 비교 분모를 맞춤.
comparison_eligible = {}
dummy_supported = {}
prediction_keys = pd.MultiIndex.from_frame(predictions[["CASE", "CONTEXT"]])
for group in groups:
    available_priors = fit_distribution.loc[
        fit_distribution["GROUP"].eq(group)
        & fit_distribution["TRAIN_0"].gt(0)
        & fit_distribution["TRAIN_1"].gt(0),
        ["CASE", "CONTEXT"],
    ].drop_duplicates()
    known_keys = pd.MultiIndex.from_frame(available_priors)
    dummy_supported[group] = pd.Series(prediction_keys.isin(known_keys), index=predictions.index)
    comparison_eligible[group] = (
        predictions[group + "__FLAG"].notna()
        & predictions[group + "__LABEL"].isin([0, 1])
        & dummy_supported[group]
    )

# 구역과 LOT마다 비교 가능한 학습 방식의 공통 행을 먼저 정함.
common_ids = {}
common_methods = {}
for group in groups:
    for lot in TEST_LOTS:
        test_rows = predictions.loc[predictions["ROLE"].eq("test") & predictions["LOT_NO"].eq(lot) & predictions["METHOD"].isin(["fixed_9_2_3", "same_grade_history", "all_history"])]
        shared = None
        available = []
        for method, block in test_rows.groupby("METHOD", sort=False):
            usable = comparison_eligible[group].loc[block.index]
            ids = set(block.loc[usable, "ROW_ID"])
            if ids:
                available.append(method)
                shared = ids if shared is None else shared.intersection(ids)
        common_ids[(group, lot)] = set() if shared is None else shared
        common_methods[(group, lot)] = available


### 6-2. Dummy 예측과 지표 계산

- `Dummy 0 고정`과 무작위 Dummy 모두 Train 라벨로만 준비함.
- 무작위 Dummy는 `DUMMY_REPEATS`번 돌림.
  - 같은 평가 행을 유지하고 난수 시드만 바꿈.
- 찾은 `1`, 놓친 `1`, 기존 `0` 중 후보로 잡힌 행을 셈.
  - 판정하지 못한 행을 놓친 `1`이나 정상 `0`에 섞지 않음.

In [ ]:
metric_rows = []
coverage_rows = []
for group in groups:
    predictions[group + "__DUMMY0"] = np.nan
    predictions[group + "__DUMMY_RANDOM0"] = np.nan
    flag_col = group + "__FLAG"
    label_col = group + "__LABEL"
    status_col = group + "__STATUS"

    for (case, method), case_rows in predictions.groupby(["CASE", "METHOD"], sort=False):
        valid = comparison_eligible[group].loc[case_rows.index]
        block = case_rows.loc[valid]
        y = block[label_col].to_numpy(dtype=int)
        dummy_contexts = []
        for context, context_rows in block.groupby("CONTEXT", sort=False):
            counts = fit_distribution.loc[
                fit_distribution["CASE"].eq(case)
                & fit_distribution["GROUP"].eq(group)
                & fit_distribution["CONTEXT"].eq(context)
            ]
            if counts.empty:
                continue
            counts = counts.iloc[0]
            n0, n1 = int(counts["TRAIN_0"]), int(counts["TRAIN_1"])
            if not n0 or not n1:
                continue

            # Train의 0/1 개수만 재현함. 라벨 순서는 Dummy의 확률에 영향을 주지 않음.
            train_y = np.repeat([0, 1], [n0, n1])
            train_x = np.zeros((len(train_y), 1), dtype="int8")
            dummy_zero = DummyClassifier(strategy="constant", constant=0)
            dummy_zero.fit(train_x, train_y)
            # Train 라벨 비율을 확률로 사용함. 예측 개수를 그 비율에 정확히 맞추지는 않음.
            dummy_random = DummyClassifier(strategy="stratified", random_state=0)
            dummy_random.fit(train_x, train_y)
            positions = block.index.get_indexer(context_rows.index)
            dummy_contexts.append((positions, dummy_zero, dummy_random))

        scopes = []
        for role, role_rows in case_rows.groupby("ROLE", sort=False):
            scopes.append(("available", role, "전체", role_rows))
            for lot, lot_rows in role_rows.groupby("LOT_NO", sort=False):
                scopes.append(("available", role, lot, lot_rows))
                if role == "test" and lot in TEST_LOTS and method in ["fixed_9_2_3", "same_grade_history", "all_history"]:
                    scopes.append(("same_test_rows", role, lot, lot_rows))

        scope_positions = []
        for scope, role, lot, rows in scopes:
            usable = comparison_eligible[group].loc[rows.index]
            selected = rows.loc[usable]
            if scope == "same_test_rows":
                selected = selected.loc[selected["ROW_ID"].isin(common_ids[(group, lot)])]
            positions = block.index.get_indexer(selected.index)
            scope_positions.append((scope, role, lot, positions))
            reason_counts = rows.loc[rows[flag_col].isna(), status_col].value_counts()
            reasons = []
            for reason, count in reason_counts.items():
                reasons.append(str(reason) + " " + str(count) + "행")
            missing_label = rows[label_col].isna() & rows[flag_col].notna()
            if missing_label.any():
                reasons.append("구역 라벨 없음 " + str(int(missing_label.sum())) + "행")
            no_prior = rows[flag_col].notna() & rows[label_col].isin([0, 1]) & ~dummy_supported[group].loc[rows.index]
            if no_prior.any():
                reasons.append("더미 학습 분포 연결 불가 " + str(int(no_prior.sum())) + "행")
            if scope == "same_test_rows":
                if not len(common_methods[(group, lot)]):
                    reasons.append("비교 가능한 학습 방식 없음")
                elif len(common_methods[(group, lot)]) == 1:
                    reasons.append("비교 가능한 학습 방식 1개")
                elif selected.empty and usable.any():
                    reasons.append("학습 방식 사이의 공통 판정 행 없음")
                common_excluded = int(usable.sum()) - len(selected)
                if common_excluded:
                    reasons.append("다른 방식과 공통인 행만 남겨 " + str(common_excluded) + "행 제외")

            selected_label1 = int(selected[label_col].eq(1).sum())
            raw_label1 = int(rows[label_col].eq(1).sum())
            coverage_rows.append({
                "SCOPE": scope,
                "CASE": case,
                "METHOD": method,
                "ROLE": role,
                "LOT_NO": lot,
                "GROUP": group,
                "RAW_ROWS": len(rows),
                "LABEL_ROWS": int(rows[label_col].isin([0, 1]).sum()),
                "RFC_SCORED_ROWS": int(rows[flag_col].notna().sum()),
                "EVALUATED_ROWS": len(selected),
                "COVERAGE": len(selected) / len(rows) if len(rows) else np.nan,
                "LABEL1_ALL": raw_label1,
                "LABEL1_EVALUATED": selected_label1,
                "LABEL1_NOT_EVALUATED": raw_label1 - selected_label1,
                "NOT_EVALUATED_ROWS": len(rows) - len(selected),
                "COMMON_METHODS": ", ".join(common_methods[(group, lot)]) if scope == "same_test_rows" else "",
                "REASON": ", ".join(reasons),
            })

        runs = [("RandomForestClassifier", -1), ("Dummy 0 고정", -1)]
        for seed in RANDOM_SEEDS:
            runs.append(("Dummy 무작위", seed))

        for model_name, seed in runs:
            pred = np.full(len(block), np.nan)
            if model_name == "RandomForestClassifier":
                pred = block[flag_col].to_numpy(dtype=float)
            else:
                for positions, dummy_zero, dummy_random in dummy_contexts:
                    predict_x = np.zeros((len(positions), 1), dtype="int8")
                    if model_name == "Dummy 0 고정":
                        pred[positions] = dummy_zero.predict(predict_x)
                    else:
                        dummy_random.set_params(random_state=seed)
                        pred[positions] = dummy_random.predict(predict_x)

            if model_name == "Dummy 0 고정":
                predictions.loc[block.index, group + "__DUMMY0"] = pred
            elif model_name == "Dummy 무작위" and seed == 0:
                predictions.loc[block.index, group + "__DUMMY_RANDOM0"] = pred

            for scope, role, lot, positions in scope_positions:
                # 더미의 Train 분포를 연결하지 못한 행도 0으로 바꾸지 않음.
                evaluated = positions[np.isfinite(pred[positions])]
                target = y[evaluated]
                predicted = pred[evaluated]
                tp = int(((target == 1) & (predicted == 1)).sum())
                fp = int(((target == 0) & (predicted == 1)).sum())
                fn = int(((target == 1) & (predicted == 0)).sum())
                tn = int(((target == 0) & (predicted == 0)).sum())
                f1_denominator = 2 * tp + fp + fn
                f1 = 2 * tp / f1_denominator if f1_denominator else 0.0
                if not len(evaluated):
                    f1 = np.nan
                metric_rows.append({
                    "SCOPE": scope,
                    "CASE": case,
                    "METHOD": method,
                    "ROLE": role,
                    "LOT_NO": lot,
                    "GROUP": group,
                    "MODEL": model_name,
                    "SEED": seed,
                    "N_ROWS": len(evaluated),
                    "N0": int((target == 0).sum()),
                    "N1": int((target == 1).sum()),
                    "TP": tp,
                    "FP": fp,
                    "FN": fn,
                    "TN": tn,
                    "CANDIDATE_ROWS": int((predicted == 1).sum()),
                    "CANDIDATE_RATE": float((predicted == 1).mean()) if len(evaluated) else np.nan,
                    "PRECISION": tp / (tp + fp) if tp + fp else np.nan,
                    "RECALL": tp / (tp + fn) if tp + fn else np.nan,
                    "F1": f1,
                })


### 6-3. 전체 구역의 기존 라벨과 예측 비교

- `해당 LOT의 기존 1`과 `평가한 기존 1`을 나눠 표시함.
  - 예측하지 못한 행을 정상 `0`이나 놓친 `1`로 세지 않음.
- `기존 0을 0으로 예측`, `찾은 1`, `잘못 잡은 0`, `놓친 1`을 함께 봄.
  - 구역마다 다른 라벨을 사용하며, 행 전체의 라벨로 합치지 않음.
- 무작위 Dummy의 F1은 반복 실행 평균임.
  - 실행별 차이는 `random_results`, 평가에서 빠진 이유는 `pending_results`에 있음.

In [ ]:
# 결과를 다시 쓰기 전에 이전 실행의 완료 표시를 지움.
completion_file = os.path.join(OUT_DIR, "run_config.json")
if os.path.isfile(completion_file):
    os.remove(completion_file)

metric_runs = pd.DataFrame(metric_rows)
coverage = pd.DataFrame(coverage_rows)
metric_runs.to_csv(os.path.join(OUT_DIR, "metrics_runs.csv"), index=False, encoding=ENC, float_format="%.7g")
coverage.to_csv(os.path.join(OUT_DIR, "coverage.csv"), index=False, encoding=ENC, float_format="%.7g")
keys = ["SCOPE", "CASE", "METHOD", "ROLE", "LOT_NO", "GROUP", "MODEL"]
counts = ["N_ROWS", "N0", "N1", "TP", "FP", "FN", "TN", "CANDIDATE_ROWS"]
rates = ["CANDIDATE_RATE", "PRECISION", "RECALL", "F1"]
summary = metric_runs.groupby(keys, sort=False)[counts].mean()
for name in rates:
    values = metric_runs.groupby(keys, sort=False)[name].agg(["mean", "std", "var", "min", "max", "count"])
    for statistic in values.columns:
        summary[name + "_" + statistic.upper()] = values[statistic]
summary["RUNS"] = metric_runs.groupby(keys, sort=False).size()
summary = summary.reset_index().merge(coverage, on=keys[:-1], how="left")
metrics = summary.loc[summary["SCOPE"].eq("available")].copy()
comparison = summary.loc[summary["SCOPE"].eq("same_test_rows")].copy()
metrics.to_csv(os.path.join(OUT_DIR, "metrics.csv"), index=False, encoding=ENC, float_format="%.7g")
comparison.to_csv(os.path.join(OUT_DIR, "comparison.csv"), index=False, encoding=ENC, float_format="%.7g")
predictions.to_csv(os.path.join(OUT_DIR, "predictions.csv"), index=False, encoding=ENC, float_format="%.17g")

show_names = {
    "METHOD": "학습 방법",
    "ROLE": "구간",
    "LOT_NO": "LOT",
    "GROUP": "구역",
    "MODEL": "모델",
    "N_ROWS": "비교 행",
    "N1": "기존 구역 1",
    "TP": "찾은 구역 1",
    "FP": "기존 구역 0 중 후보",
    "FN": "놓친 구역 1",
    "TN": "기존 구역 0 중 미검출",
    "PRECISION_MEAN": "Precision",
    "RECALL_MEAN": "Recall",
    "F1_MEAN": "F1",
    "CANDIDATE_RATE_MEAN": "후보 비율",
    "NOT_EVALUATED_ROWS": "비교에서 빠진 행",
    "LABEL1_NOT_EVALUATED": "비교에서 빠진 구역 1",
    "REASON": "보류 또는 비교 제외 이유",
}
# 전체 구역의 결과를 먼저 봄. 구역 선택 때문에 결과가 숨겨지지 않게 함.
result_keys = ["CASE", "METHOD", "ROLE", "LOT_NO", "GROUP"]
result_columns = result_keys + [
    "N_ROWS", "LABEL1_ALL", "N1", "TP", "TN", "FP", "FN",
    "PRECISION_MEAN", "RECALL_MEAN", "F1_MEAN", "COVERAGE",
    "LABEL1_NOT_EVALUATED", "REASON",
]
all_results = metrics.loc[
    metrics["MODEL"].eq("RandomForestClassifier") & metrics["LOT_NO"].ne("전체"),
    result_columns,
].copy()

for model, column in [("Dummy 0 고정", "DUMMY0_F1"), ("Dummy 무작위", "DUMMY_RANDOM_F1")]:
    dummy_result = metrics.loc[
        metrics["MODEL"].eq(model) & metrics["LOT_NO"].ne("전체"),
        result_keys + ["F1_MEAN"],
    ].rename(columns={"F1_MEAN": column})
    all_results = all_results.merge(dummy_result, on=result_keys, how="left")

all_results["RESULT_STATUS"] = "평가함"
all_results.loc[all_results["N_ROWS"].eq(0), "RESULT_STATUS"] = "평가 못 함"
all_results.to_csv(os.path.join(OUT_DIR, "all_group_results.csv"), index=False, encoding=ENC)

show_names.update({
    "LABEL1_ALL": "해당 LOT의 기존 1",
    "N1": "평가한 기존 1",
    "TN": "기존 0을 0으로 예측",
    "F1_MEAN": "분류 모델 F1",
    "DUMMY0_F1": "Dummy 0 F1",
    "DUMMY_RANDOM_F1": "무작위 Dummy F1 평균",
    "COVERAGE": "평가한 비율",
    "RESULT_STATUS": "평가 상태",
})
show_columns = [
    "METHOD", "ROLE", "LOT_NO", "GROUP", "RESULT_STATUS",
    "N_ROWS", "LABEL1_ALL", "N1", "TP", "TN", "FP", "FN",
    "PRECISION_MEAN", "RECALL_MEAN", "F1_MEAN",
    "DUMMY0_F1", "DUMMY_RANDOM_F1", "LABEL1_NOT_EVALUATED",
]
print("모든 LOT, 모든 구역: 기존 0과 1을 각각 얼마나 맞혔는지 비교")
print("평가한 기존 1이 0이어도, 해당 LOT의 기존 1은 있을 수 있음. 평가 상태와 비교에서 빠진 1을 함께 봄.")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(all_results[show_columns].replace({"METHOD": method_names}).rename(columns=show_names).round(4).reset_index(drop=True))

print("117, 121, 133: 학습 방법끼리도 같은 행에서 비교한 결과")
comparison_columns = [
    "METHOD", "ROLE", "LOT_NO", "GROUP", "MODEL", "N_ROWS", "N1",
    "TP", "TN", "FP", "FN", "PRECISION_MEAN", "RECALL_MEAN", "F1_MEAN",
]
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(comparison[comparison_columns].replace({"METHOD": method_names}).rename(columns=show_names).round(4).reset_index(drop=True))

# 무작위 예측의 차이와 보류 이유는 모든 구역을 포함해 별도 표로 남김.
# 위 요약에는 평균만 보이며, 아래 변수에서 표준편차와 최소, 최대를 확인할 수 있음.
random_results = metrics.loc[
    metrics["MODEL"].eq("Dummy 무작위") & metrics["LOT_NO"].ne("전체"),
    result_keys + ["N_ROWS", "F1_MEAN", "F1_STD", "F1_MIN", "F1_MAX", "F1_COUNT"],
].copy()
pending_results = coverage.loc[
    coverage["SCOPE"].eq("available") & coverage["LOT_NO"].ne("전체")
    & coverage["NOT_EVALUATED_ROWS"].gt(0)
].copy()
print("무작위 예측의 반복 차이: random_results / 평가하지 못한 이유: pending_results")


### 7. 모든 모델 결과를 Figure 하나에서 보기

- 학습 방법별로 한 줄씩 배치함.
  - 각 그림의 세로축은 `LOT`, 가로축은 `구역`임.
- 세 모델의 F1과 분류 모델이 잘못 잡거나 놓친 수를 같은 위치에서 봄.
  - `평가한 비율`도 함께 확인함.
- `보류`는 평가하지 못한 경우이며 성능 `0`이 아님.
  - 흰색 `—`는 해당 실험에서 평가하는 LOT가 아님.
  - 실제 `1`과 예측 `1`이 모두 없으면 F1은 `정의 불가`로 표시함.
- 구역별 센서 그래프를 여러 장 저장하지 않고 비교 그림 한 장만 저장함.
  - 세 방법은 학습 자료가 다르므로, 같은 행끼리 비교할 때는 위 `comparison` 표를 봄.

In [ ]:
# 모든 구역을 같은 순서로 놓고, 한 실행의 결과를 Figure 하나에 모음.
# 원래 라벨과 모델 예측은 바꾸지 않고 저장한 지표만 그림으로 표시함.
plot_methods = ["fixed_9_2_3", "same_grade_history", "all_history"]
if metrics["METHOD"].eq("first_lot_60_20_20").any():
    plot_methods.append("first_lot_60_20_20")

if metrics["METHOD"].eq("within_lot_60_20_20").any():
    plot_methods.append("within_lot_60_20_20")

plot_groups = list(groups)
plot_lots = LOTS
panels = [
    ("RandomForestClassifier", "F1_MEAN", "분류 모델 F1", "Blues"),
    ("Dummy 0 고정", "F1_MEAN", "전부 0인 Dummy F1", "Blues"),
    ("Dummy 무작위", "F1_MEAN", "무작위 Dummy F1 평균", "Blues"),
    ("RandomForestClassifier", "FP", "기존 0을 1로 예측한 수", "Oranges"),
    ("RandomForestClassifier", "FN", "기존 1을 0으로 예측한 수", "Oranges"),
    ("RandomForestClassifier", "COVERAGE", "전체 기록 중 평가한 비율", "Greens"),
]

fig, axes = plt.subplots(
    len(plot_methods), len(panels),
    figsize=(36, 6.1 * len(plot_methods)),
    squeeze=False,
)

# 횟수 그림은 학습 방법이 달라도 같은 색 범위를 사용함.
count_limits = {}
for column in ["FP", "FN"]:
    count_limits[column] = max(1, metrics.loc[
        metrics["LOT_NO"].ne("전체") & metrics["MODEL"].eq("RandomForestClassifier"), column
    ].max())

for method_index, method in enumerate(plot_methods):
    for panel_index, (model, column, title, color_map) in enumerate(panels):
        axis = axes[method_index, panel_index]
        rows = metrics.loc[
            metrics["METHOD"].eq(method) & metrics["MODEL"].eq(model)
            & metrics["LOT_NO"].ne("전체")
        ].copy()
        if method in ["first_lot_60_20_20", "within_lot_60_20_20"]:
            rows = rows.loc[rows["ROLE"].eq("test")].copy()
        rows["LOT_NO"] = rows["LOT_NO"].astype(str)
        values = rows.pivot(index="LOT_NO", columns="GROUP", values=column)
        values = values.reindex(index=plot_lots, columns=plot_groups)
        compared = rows.pivot(index="LOT_NO", columns="GROUP", values="N_ROWS")
        compared = compared.reindex(index=plot_lots, columns=plot_groups)
        exists = rows.pivot(index="LOT_NO", columns="GROUP", values="RAW_ROWS")
        exists = exists.reindex(index=plot_lots, columns=plot_groups)

        # 비교하지 못한 칸은 성능 0이나 오류 0으로 표시하지 않음.
        # 평가 비율만 0% 그대로 보여, 평가하지 못했다는 사실을 남김.
        if column != "COVERAGE":
            values = values.where(compared.gt(0))
        if column == "F1_MEAN":
            denominator = rows["TP"] * 2 + rows["FP"] + rows["FN"]
            rows["F1_DEFINED"] = denominator.gt(0)
            defined = rows.pivot(index="LOT_NO", columns="GROUP", values="F1_DEFINED")
            values = values.where(defined.reindex(index=plot_lots, columns=plot_groups).eq(True))

        colors = plt.get_cmap(color_map).copy()
        colors.set_bad("#e5e7eb")
        maximum = count_limits[column] if column in count_limits else 1
        heat = axis.imshow(values.to_numpy(dtype=float), cmap=colors, vmin=0, vmax=maximum, aspect="auto")

        for lot_index, lot in enumerate(plot_lots):
            for group_index, group in enumerate(plot_groups):
                value = values.loc[lot, group]
                if pd.isna(exists.loc[lot, group]):
                    text = "—"
                    axis.add_patch(plt.Rectangle(
                        (group_index - 0.5, lot_index - 0.5), 1, 1,
                        facecolor="white", edgecolor="none",
                    ))
                elif column != "COVERAGE" and compared.loc[lot, group] == 0:
                    text = "보류"
                elif pd.isna(value):
                    text = "정의\n불가"
                elif column == "COVERAGE":
                    text = str(round(value * 100)) + "%"
                elif column == "F1_MEAN":
                    text = format(value, ".2f")
                else:
                    text = str(int(round(value)))

                text_color = "white" if pd.notna(value) and value > maximum * 0.65 else "#243447"
                axis.text(group_index, lot_index, text, ha="center", va="center", fontsize=7.5, color=text_color)

        axis.set_xticks(range(len(plot_groups)), plot_groups, rotation=55, ha="right", fontsize=9)
        axis.set_yticks(range(len(plot_lots)), [lot[-3:] for lot in plot_lots], fontsize=9)
        axis.set_title(method_names[method] + "\n" + title, fontsize=13, pad=12)
        axis.set_ylabel("LOT 번호 끝 3자리")
        axis.set_xticks(np.arange(len(plot_groups) + 1) - 0.5, minor=True)
        axis.set_yticks(np.arange(len(plot_lots) + 1) - 0.5, minor=True)
        axis.grid(which="minor", color="white", linewidth=0.7)
        axis.tick_params(which="minor", bottom=False, left=False)
        fig.colorbar(heat, ax=axis, fraction=0.035, pad=0.02)

fig.suptitle(DATA_FILE + " | 모든 LOT와 구역의 기존 라벨, 모델 예측 비교", fontsize=22, y=0.985)
fig.text(
    0.035, 0.945,
    "각 칸은 같은 LOT와 같은 구역임. F1은 높을수록 기존 라벨과 잘 맞음. 주황색 수는 적을수록 좋음.\n"
    "흰색 —: 해당 실험의 평가 대상 아님. 회색 보류: 평가하지 못함. 정의 불가: 실제 1과 예측 1이 모두 없어 F1을 계산할 수 없음.",
    fontsize=12, va="top",
)
fig.text(
    0.035, 0.018,
    "고정 9/2/3: 108, 114는 Valid / 117, 121, 133은 Test. 이전 강종 방법: 각 LOT보다 앞선 자료로 학습함. LOT 내부 분할은 뒤 20% Test만 그림에 표시함.\n"
    "각 학습 방법 안에서 세 모델은 같은 행을 비교함. 학습 방법끼리의 공통 Test 행 비교는 위 comparison 표를 봄.\n"
    "무작위 Dummy는 " + str(DUMMY_REPEATS) + "회 F1 평균임. 기존 라벨 1은 이상 후보이며 고장 정답이 아님.",
    fontsize=11,
)
fig.subplots_adjust(left=0.035, right=0.985, top=0.885, bottom=0.095, hspace=0.42, wspace=0.32)
figure_path = os.path.join(FIG_DIR, "rfc_all_lots_all_groups.png")
fig.savefig(figure_path, dpi=150, facecolor="white")
plt.show()
plt.close(fig)
print("그림 한 장:", figure_path)


### 8. 실행에 사용한 파일과 옵션 저장

- `run_config.json`에 CSV 이름, 파일 내용 식별값, 학습 옵션을 남김
  - CSV를 바꾸거나 다시 실행하면 새 결과 폴더를 만듦
- `fit_rows.csv.gz`는 학습에 넣으려던 행과 실제 학습 여부를 남긴 CSV임
  - `STATUS`가 `학습 완료`인 행만 실제 모델 학습에 사용했음
  - 한 종류의 라벨만 있어 학습을 보류한 행도 그 이유와 함께 남김
  - 각 구역을 확인할 때마다 파일 끝에 추가 저장함
  - 내용은 같고 파일 크기만 줄임
  - `pd.read_csv`로 그대로 읽을 수 있음
- 구역 라벨을 얼마나 재현했는지 확인하는 실험임
  - Z-score 규칙에서 만든 라벨이므로 실제 고장 탐지 성능과 구분함
  - 학습하지 않은 강종과 작업 조건, 한쪽 라벨이 없는 구역은 보류함
- 정비 종료 기록과 팀의 제외 판단은 사후 정보임
  - 실제 운전 중에 바로 경보를 내보낸 실험은 아님

In [ ]:
input_columns_by_group = {}
for group, group_input in group_inputs.items():
    input_columns_by_group[group] = group_input.columns.tolist()

config = {
    "input_file": DATA_FILE,
    "input_sha256": input_hashes,
    "data_kind": DATA_KIND,
    "run_id": RUN_ID,
    "groups": groups,
    "input_columns_by_group": input_columns_by_group,
    "train_lots": TRAIN_LOTS,
    "valid_lots": VALID_LOTS,
    "test_lots": TEST_LOTS,
    "run_first_lot": RUN_FIRST_LOT,
    "run_no_history_lot": RUN_NO_HISTORY_LOT,
    "parameters": {
        "n_estimators": N_ESTIMATORS,
        "max_depth": MAX_DEPTH,
        "min_samples_leaf": MIN_SAMPLES_LEAF,
        "max_features": "sqrt",
        "bootstrap": True,
        "class_weight": CLASS_WEIGHT,
        "random_state": SEED,
        "n_jobs": 2,
        "probability_threshold": PROBABILITY_THRESHOLD,
        "dummy_repeats": DUMMY_REPEATS,
    },
    "reference": {
        "minimum_rows": MIN_REFERENCE_ROWS,
        "minimum_observed_minutes": MIN_REFERENCE_MINUTES,
        "fit_data": "Train known group labels and valid group input only",
        "grade_center": "Train median",
        "grade_scale": "Train IQR, fallback std ddof0, fallback1",
        "raw_window_seconds": 60,
        "raw_window_closed": "left",
        "raw_min_past_rows": MIN_PAST_ROWS,
        "raw_gap_seconds": MAX_GAP_SECONDS,
        "minute_features": "precomputed raw60s group feature mean",
    },
    "label_meaning": "group candidate rule, not confirmed failure",
    "label_created_with_complete_lot": True,
    "future_observations_in_feature_inputs": False,
    "retrospective_operating_context": True,
    "evaluation_labels_used_for_fit": False,
    "test_used_to_choose_threshold": False,
    "sampling_to_20pct_applied": False,
    "hold_is_zero": False,
    "fit_rows_file": "fit_rows.csv.gz",
    "source_notebook": "notebooks/sayletskin/04_random_forest_classifier.ipynb",
    "versions": {
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "issue": 91,
}
with open(os.path.join(OUT_DIR, "run_config.json"), "w", encoding="utf-8") as file:
    json.dump(config, file, ensure_ascii=False, indent=2)
print("결과:", OUT_DIR)
print("그림:", FIG_DIR)
